# File Input/Output

> ### Learning Objectives
>
> By the end of this chapter you should be able to work with:
>
> - How running program data (volatile memory) differs from persistent storage (drives, USB, cloud)
> - File systems: files, folders/directories, extensions, and paths
> - Data file formats: human-readable text versus binary, and common examples of each
> - Opening and closing files with `open()` (read flag) and `close()`, and why releasing the resource matters
> - Reading a file line by line with a `for` loop over the file object
> - Why each line includes a trailing newline, and using `.strip()` to remove it
> - Converting text read from a file to numeric types for computation
> - Splitting lines of a tabular/delimited file with `split()`
> - Writing to files with write (overwrite) and append modes
> - Building output strings with `.join()`
> - Combining `.join()` with a list comprehension to write non-string values (e.g., integers)
> - Absolute versus relative paths (`..` and `.`)
> - OS path-separator differences and using `os.path.join` for cross-platform paths

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SETUP — run this cell first.
#
#  It draws every figure used in this chapter and switches the notebook into
#  "show me every result" mode.  Everything it needs is right here: nothing to
#  install, nothing to download, no other files required.
#
#  (Curious what a figure is made of?  The drawing code is all below.)
# ══════════════════════════════════════════════════════════════════════
import io

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Circle
from matplotlib.lines import Line2D
from IPython.display import Image, display
from IPython.core.interactiveshell import InteractiveShell

# echo the value of *every* expression in a cell, the way the Python prompt
# does — many examples in this book show several results at once
InteractiveShell.ast_node_interactivity = "all"

# ---------------------------------------------------------------- runtime ---
_SIZES = {}
_WIDTHS = {}
_FIGURES = {}


def _render(name):
    fig, ax = plt.subplots(figsize=_SIZES.get(name, (6.4, 4.4)), dpi=110)
    globals()["draw_" + name](ax)
    fig.tight_layout(pad=0.3)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return buf.getvalue()


def show(name, width=None):
    """Display one of this chapter's figures."""
    display(Image(_FIGURES[name], width=width or _WIDTHS.get(name, 560)))


for _n in []:
    _FIGURES[_n] = _render(_n)


def write_sample_files():
    """Create the data files this chapter reads. Safe to run repeatedly."""
    files = {
        # one temperature observation per line (a "list file")
        "temperatures.txt": "-2.7\n-1.8\n0.3\n2.4\n3.5\n5.9\n",
        # one movie title per line (a "list file")
        "movietitles.txt": (
            "The Fellowship of the Ring\n"
            "The Two Towers\n"
            "The Return of the King\n"
        ),
        # weather stations: a four-digit station ID then six temperatures,
        # tab-delimited (a "tabular file")
        "temptable.txt": (
            "1783\t22\t25\t27\t28\t21\t19\n"
            "2214\t-4\t2\t6\t7\t6\t0\n"
            "9934\t-40\t-32\t-26\t-21\t-24\t-32\n"
            "5538\t15\t17\t21\t22\t23\t19\n"
        ),
    }
    for name, text in files.items():
        with open(name, "w") as f:
            f.write(text)
    return sorted(files)

write_sample_files()
print("Setup complete \u2014 0 figure(s) ready.")


Up to this point, the only mechanisms we have used for data **input** into our programs is to either code the data right into our program as literal data (this is sometimes called *hard-coding* the data) or ask the user to enter input from the console.  In this chapter, we look at how to obtain input data stored in files.  Similarly, the only way we have seen our programs produce **output** is to print to the console.  In this chapter we will also look at how to write output to a file.

### Data File Formats

The term *file format* refers to the way in which data is organized in a file.  There are two main types of file formats: text file formats and binary file formats.

Text file formats are readable by humans.  You can open them in any text editor and see the data inside and how it is organized.  In text files, numbers are stored as strings of digits.  A text file containing data about cities and their average annual high temperatures might look like this:

```

Saskatoon   9 
Vancouver   14
Winnipeg    9
Toronto     13

```

Binary file formats are generally not readable by humans because the data is binary-encoded.  Such files generally do not contain any meaningful whitespace such as spaces or newlines and appear as gibberish when viewed in a text editor.  In a binary file format, numbers are stored in binary (base 2) format, in groups of 8-bits (a byte).  A number might be comprised of the bits in one, two, or four consecutive bytes.   If we stored the temperature data, above, in a binary file, it might look something like this when we load it into a text editor:
Binary files are typically more compact, use less disk space, and are used frequently in commercial applications and games. Since CS-1910 is an introductory course, we will not be using any binary file formats, only text file formats. You will learn more about reading and writing binary files in CS-1920 and CS-2910.

#### Common Text File Formats

In this section, we review two typical ways in which we might organize data in a text file.

##### List Files: One Data item Per Line

A *list file* consists of one data item per line, and usually each line contains the same type of data.  List files are very simple to read into a program since each line of the file contains one data item, and most programming languages have built-in functions for reading one line from a file.  An example of a list file might be observations of temperature recorded over a single day:

```

-2.7
-1.8
0.3
2.4
3.5
5.9

```

##### Tabular Files: One Group of Related Data Items Per Line

A *tabular file* format is one where there is a fixed number of data items per line.  We can think of such a file as a table, because it will have a certain number of rows (lines) and a certain number of columns (data items per line).

The data items on a line may be different types, but typically each column of data is all of the same type, that is, the $n$-th piece of data on each line is of the same type.  Data items on a line might be separated by spaces, or another character, such as a comma.  Whatever character is used to indicate separation of data items on a line of the file is called the file's *delimiter*.  It delimits (separates) one data item from the next.  Here is an example of a tabular text file, delimited by commas, where each line holds data from an entry in the database from Section :

```

Homer J. Simpson,Homer,Simpson,J,742 Evergreen Terrace,Springfield,Unknown,USA
Charles M. Burns,Charles,Burns,M,1000 Mammon Ave.,Springfield,Unknown,USA
Ned Flanders,Ned,Flanders,,744 Evergreen Terrace,Springfield,Unknown,USA

```

Each line of this file holds the data for one database entry, and contains exactly 8 data items, separated by commas.  The first data item on each line is the key for a database entry, and the remaining data items are the data items in the database record associated with the key.  Note how the fourth data item of the third database entry is empty since there is nothing between the commas.

If our data items themselves do not contain spaces, we can use whitespace as a delimiter, which makes the text file look more like a table.  Here is an example of a tabular datafile that stores weather observations taken every four hours for different weather stations on one specific day of the year where each weather station is identified by a four-digit ID number:

```

1783	22 	25 	27 	28 	21 	19
2214	-4	2	6	7	6	0
9934	-40	-32	-26	-21	-24	-32
5538	15	17	21	22	23	19

```

The first column contains the weather station ID number, and the remaining columns store temperature observations.   Since observations are every four hours, there are six such columns.

##### Other Formats

Any format you can think of is theoretically possible, but you might have to write custom code that can process unconventional formats.

### File Objects in Python -- Open and Closing Files

In Python we interact with files on the disk via an abstraction.  We can ask Python to return an object that allows us to interact with a data file on disk.  This is called *opening* a file.  We can open a file and obtain an object for that file using Python's built-in `open` function.  The `open` function returns an object that contains methods that allow us to read and write data to or from a file.  Suppose the table of temperature data, above, is stored in a file called `temperatures.txt`.  We can open it like this:

In [ ]:
f = open("temperatures.txt", "r")

The first argument to `open` is a string containing the name of the file to be opened --- this can be any valid pathname (see another section for more details on pathnames).   The second argument string is the *mode*.  Here we are using the file mode `'r'`, to indicate that we want to **read** from the file.  Later we'll see how to write to files using the `'w'` mode.
Now `f` is an object that contains methods that allow us to manipulate the file.  This is a nice abstraction because we can work with the file just by calling methods of `f` and we don't need to know how disks and file systems work.  One other interesting thing about file objects is that they behave as sequences, which means we can use them in places where we could use sequences!  We'll see how this works in the next few sections.

Before we move on, we must note that once a file is opened, it must be *closed* again when you are done with it.  If `f` refers to a file object created with `open`, it is closed by calling the `close` method of `f`:

In [ ]:
f.close()

Once you call `f.close()`, `f` can no longer be used to manipulate the file; trying to do so will result in an error message.
If you forget to close a file that was opened in **read** mode, usually nothing bad will happen, although you really should always do it.  If you forget to close a file that was opened in **write** mode, it is possible that the data you wrote to the file will not actually be written, and that is very bad!

### Reading Text Files

In the previous section, we mentioned that file objects returned by the `open` function behave like sequences.  In particular, they behave like sequences of strings, where each string is a line of the file.  This means that we can iterate over the lines of a file just like we can iterate over the elements of a list!

#### Reading List Files

List files are pretty easy to deal with since each line of a file contains a single data item and, as we have already mentioned, we can access each line of a file as a string easily.

Suppose we have a file called `movietitles.txt` which contains one movie title per line.  We can read the movie titles from the file and store them in a Python list like this:

In [ ]:
# Open the file for reading
f = open("movietitles.txt", "r")

# create an empty list
titles = []

# iterate over each line of the file
for line in f:
	# append the next line (movie title) to the list
	titles.append(line)
	
# close the file
f.close()

If `movietitles.txt` contains the following data:

```

The Fellowship of the Ring
The Two Towers
The Return of the King

```

Then the above code will result in `titles` referring to the list:

```

["The Fellowship of the Ring\n", "The Two Towers\n", "The Return of the King\n"]

```

Hey, wait, that's weird.  What are those `\textbackslash n`'s at the end of each string in the list?  Those are *newline* characters; they are invisible characters that mark the end of each line in a text file, and therefore are included in the string that comprises a line of the file.  In Python,  `\textbackslash n` represents the newline character.  Even though it is represented by two characters, \\ and `n`, it is actually a single character.  It is represented this way so that we can see it because normally it is invisible since it is not associated with any symbol.

Usually we don't want newline characters in our strings.  We can remove them by calling the string method `rstrip`.  If `s` refers to a string, then `s.rstrip()` returns a copy of `s` that has all of whitespace at the end of the string, including spaces and newlines, removed.  Revising our loop in the previous code to this:

In [ ]:
f = open("movietitles.txt", "r")
titles = []
# iterate over each line of the file
for line in f:
	# append the next line (movie title) to the list
	titles.append(line.rstrip())
f.close()

results in `titles` referring to the list:

```

["The Fellowship of the Ring", "The Two Towers", "The Return of the King"]

```

Another way to create a list of the strings from the lines in a file is to use the `list` function to convert the sequence of lines from the file object `f` to a list.  Then we can use a list comprehension to remove the newlines:

In [ ]:
f = open("movietitles.txt", "r")
titles = list(f)
titles = [ t.rstrip() for t in titles ]
f.close()

The result of this code is the same as the previous code listing.

What if we have a list file of numbers?  This would seem to be a problem if file objects can only return each line as a string because we would want to read in a file of numbers and store them as numbers, not strings.   We can use the built-in functions `int` or `float` to convert strings to numbers.  For example `int("42")` returns the integer 42, and `float("64.9")` returns the floating point value 62.9.  If you use `int` or `float` on a string that doesn't represent a number of the appropriate type, Python will respond with a `ValueError`.  We could read the list file containing temperature data at the beginning of another section and store the data as a list of floats like this:

In [ ]:
f = open("temperatures.txt", "r")
temps = []
for line in f:
	temps.append(float(line))
f.close()

or equivalently:

In [ ]:
f = open("temperatures.txt", "r")
temps = list(f)
temps = [float(t) for t in temps]
f.close()

Both programs here would cause `temps` to refer to the list:

```

[-2.7, -1.8, 0.3, 2.4, 3.5, 5.9]

```

#### Reading Tabular Files

Reading tabular files is almost the same as reading list files.  The main difference is that we have to separate the data items on each line.  Remember that the data items on each line are separated by a delimiter.  String objects have a `split` method which returns a list of strings consisting of the individual strings that occur between a specific delimiter character.  For example, the string `"The king in the north."` can be separated into individual words like this:

In [ ]:
my_string = "The king in the north."
words = my_string.split()

This results in `words` referring to the list:

```

["The", "king", "in", "the", "north."]

```

If we want to split a string based on a delimiter other than whitespace, we just pass the desired delimiter to `split` as an argument.  Here's how we can obtain a list of strings from a string delimited by commas:

In [ ]:
my_string = "42,38,27,99,55"
numbers = my_string.split(",")

This results in `numbers` referring to the list

```

["42", "38", "27", "99", "55"]

```

They're still strings, but we've already seen how we can use a list comprehension to convert this to a list of integers or floats.

We can obtain the lines of a tabular data file in the same way that we obtained lines for list files, but then we have to use `split` to divide up each line into its individual data items.  A common way to store the data from a tabular file in Python is a list in which each data item is another list that contains the data items from one line of the file, i.e. a list of lists.  Recall the temperature data in the tabular file we saw in Section :

```

1783	22 	25 	27 	28 	21 	19
2214	-4	2	6	7	6	0
9934	-40	-32	-26	-21	-24	-32
5538	15	17	21	22	23	19

```

The following code reads this data and stores it as a list of lists of integers:

In [ ]:
f = open("temptable.txt")
stations = []
for line in f:
	stations.append([int(n) for n in line.split()])
f.close()

Observe how we read each line, split it (using whitespace as a delimiter), then convert the resulting list of strings into a list of integers, then append that list to the list `stations`.
This causes `stations` to refer to the list:

```

[
  [1783, 22, 25, 27, 28, 21, 19], 
  [2214, -4, 2, 6, 7, 6, 0], 
  [9934, -40, -32, -26, -21, -24, -32], 
  [5538, 15, 17, 21, 22, 23, 19], 
]

```

Observe that `stations[i]` refers to the data in the `i`-th line of the file, and `stations[i][j]` refers to the item in the `j`-th column of the `i`-th line of the file. Thus, `stations[1][4]` refers to the file data found at the fifth column of the second line, which is 7.

### Writing Text Files

To write to a file, you have to open it in **write** mode:

In [ ]:
f = open("file_to_write.txt", "w")

If a file is opened in write mode, and a file of the same name already exists, then the existing file is destroyed, and a new file of the same name replaces it.  If the file opened for writing does not exist yet, it is created.

It is possible to write data at the end of an existing file without destroying it.  To do so, open the file in **append** mode:

In [ ]:
f = open("file_to_write.txt", "a")

#### The `write()` method.

Writing data to text files is very similar to printing to the console.  First you have to open a file in **write** or **append** mode.  Then, instead of using the `print` function, you use the `write` method of the resulting file object.  If the variable `f` refers to a file object, and the file was opened in **write** mode, then the code

> *A fragment for illustration — it is not complete enough to run.*
```python
f.write(string)
```

writes the the string `*string*` to the file.  The `write` method does not write a newline character to the file unless the string given as an argument includes one. Note that this behaviour is different from the `print` function which, by default, always outputs a newline after printing its argument.

#### Writing List Files

List files can be written by writing each data item followed by a new line.  If we have a list of strings, we can write those strings, one per line, to a file called `shoppinglist.txt` like this:

In [ ]:
ingredients = ["eggs", "milk", "flour", "yeast"]
f = open("shoppinglist.txt", "w")
for i in ingredients:
	f.write(i + "\n")
f.close()

This code iterates over each item in the list `ingredients`, and writes it to the file.  Note how we concatenate each item in the list with a newline before writing it so that each string appears on its own line.  The resulting file looks like this:

```

eggs
milk
flour
yeast

```

If the items we are writing are not strings, we have to convert them to strings because the `write` method can only write strings to files.  We can do this using the built-in `str` function which converts its argument to a string, if possible.  Here's how we would write a list of integers to a file, one per line:

In [ ]:
ingredients = [99, 88, 77, 66, 55]
f = open("numbers.txt", "w")
for i in ingredients:
	f.write(str(i) + "\n")
f.close()

Note how the integer `i` is converted to a string prior to concatenating it with a newline.

#### Writing Tabular Files

To write a tabular file, a typical strategy is to construct a string consisting of one line of the tabular file to be written, and then write it.  This is done by combining the data items to appear on that line into a single string, separated by the appropriate delimiter.  Just as we had a method, `split`, that could separate a delimited string, we have one that can construct a delimited string from a list of individual data items.   String objects have a method called `join`.  This method takes a list as an argument and returns a new string that consists of the items in the list separated by the original string.  Remember: the string on which we call the `join` is the separator, and the list provided as an argument to `join` contains the data items to combine.

Suppose we have a list of numbers `numbers` which should all appear on one line of a tabular file, separated by commas. We can construct the appropriate string to write to the file like this:

In [ ]:
numbers = [42, 24, 87, 21, 76]
line = ",".join([str(x) for x in numbers])
print(line)

This produces the following output **string**:

```

42,24,87,21,76

```

Look what's happening here.  The list comprehension `[str(x) for x in numbers]` converts the list of integers `numbers` into a list of strings.  This list is then passed to the join method of the string object ",".  This causes the elements of the list to be concatenated, separated by the string `","`.  The result is that `line` refers to the **string** `"42,24,87,21,76"` which is then output by the print statement.

Putting all of this together, suppose we had a list of lists.  We could write all the data items of each list to a tabular file like this:

In [ ]:
# a list of lists.  We've seen this temperature data before.
data = [
  [1783, 22, 25, 27, 28, 21, 19], 
  [2214, -4, 2, 6, 7, 6, 0], 
  [9934, -40, -32, -26, -21, -24, -32], 
  [5538, 15, 17, 21, 22, 23, 19], 
]
f = open("temperaturedata.txt", "w")
for station in data:
	f.write(",".join([str(i) for i in station])+"\n")
f.close()

For each list of **integers** `station` in `data`, we use a list comprehension to convert the items in `station` to **strings** and put them into a new list, then `join` this list of strings into a single string with a comma as a separator, then add a newline to the end of the resulting string, and write it to the file.  This results in a tabular text file that looks like this:

```

1783,22,25,27,28,21,19
2214,-4,2,6,7,6,0
9934,-40,-32,-26,-21,-24,-32
5538,15,17,21,22,23,19

```

### Pathnames

Even if you've never programmed a computer before, but rather, only used one, you probably already know something about pathnames.
*Pathnames* are strings that refer to files.  When we use the `open` function, we said back earlier that we need to pass a *pathname* as an argument to `open` to tell it which file to open.  If you want to open a file in the same folder as your Python program, you only need to specify the file's name as a string, like `"temperatures.txt"` or `"reallycooldata.csv"`.  If the file exists somewhere else you need to give a full pathname that also specifies the folder that the file resides in.  The mechanism for doing this depends on your computer's operating system.    On Windows, folder names are separated by a backslash, and the whole pathname might be preceded by a drive letter:
`"C:\Users\Chris\My Documents\awesomedata.csv"`
On Mac and Linux, folder names are separated in a pathname by a forward slash:
`"/home/chris/Documents/awesomedata.csv"`
These are examples of *absolute paths* because they specify the entire path to the file beginning at the root folder.  You can also use *relative paths* which specify the path to a file beginning from the folder that your Python program is in, such as:
`"../../experiment/data/specialdata.txt"`
The folder name `".."` means "parent folder".  So the above path means go "up" two folders, then go into the `experiment/data` folder, and find `specialdata.txt` there.

The different folder separator for Windows and Linux/Mac means we have to be a little careful if we want our programs to work on **all** operating systems.  Fortunately, Python has a module for that.  The `os.path` module has methods for constructing pathnames using the appropriate folder separator for the operating system you are currently running on.

The method `os.path.join()` method can be used to concatenate folder and file names using the correct separator.  The variable `os.sep` also refers to the correct separator. Examples:

In [ ]:
import os

# An absolute path:
filename = os.path.join(os.sep, "home", "chris",
                        "Documents", "awesomedata")
print(filename)

# a relative path:
filename = os.path.join("..", "experiment", "data",
                        "specialdata.txt")
print(filename)

Try this on different operating systems and you'll notices differences in the string referred to by `filename`.  Experiment on your own with `os.path.join()` until you're comfortable with how it works.

There are lots of other ways of creating and manipulating paths in the `os.path` module that are outside the scope of the course, but you can learn more about them here if you are interested:  <https://docs.python.org/3.5/library/os.path.html>.

> **Optional Trivia Challenge**
>
> Why do drive letters on Windows operating systems start at 'C' and not 'A'?